In [1]:
# 安装所需的数据分析和Excel处理库
!pip install pandas openpyxl numpy --upgrade

Requirement already up-to-date: pandas in c:\programdata\anaconda3\lib\site-packages (0.24.2)
Requirement already up-to-date: openpyxl in c:\programdata\anaconda3\lib\site-packages (2.6.1)
Requirement already up-to-date: numpy in c:\programdata\anaconda3\lib\site-packages (1.16.2)


Retrying (Retry(total=4, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ProtocolError('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None))': /simple/pandas/
Retrying (Retry(total=3, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='pypi.org', port=443): Read timed out. (read timeout=15)")': /simple/pandas/


In [ ]:
# 一次性解决所有依赖问题（必须先运行！）
!pip install --upgrade pandas openpyxl xlrd

In [2]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

def full_fill_format_all(excel_path, output_path, column_width=15, row_height=30):
    """
    完整功能：
    1. 全量补充所有子表信息到表单数据
    2. 统一所有列宽度（column_width）
    3. 统一所有数据行高度（row_height）
    4. 优化单元格样式（自动换行、居中对齐）
    """
    print("="*70)
    print(f"开始执行：信息全量补充 + 列宽统一({column_width}) + 行高统一({row_height})")
    print("="*70)

    # ---------------------- 1. 数据表映射关系 ----------------------
    sheet_mapping = {
        "受教育经历（从高中开始）（请在sheet名称为“受教育经历（从高中开始）”中填写）": {
            "sub_sheet": "受教育经历（从高中开始）",
            "fields": ["学校名称", "起始日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", 
                      "终止日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", "专业", "学历/学位"]
        },
        "入学前主要获奖情况（请在sheet名称为“入学前主要获奖情况”中填写）": {
            "sub_sheet": "入学前主要获奖情况",
            "fields": ["奖励名称", "奖励级别", "获得日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy"]
        },
        "工作经历（请在sheet名称为“工作经历”中填写）": {
            "sub_sheet": "工作经历",
            "fields": ["任职单位", "起始日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", 
                      "终止日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", "职务", "任职地点"]
        },
        "家庭成员信息（请在sheet名称为“家庭成员信息”中填写）": {
            "sub_sheet": "家庭成员信息",
            "fields": ["姓名", "与本人关系", "工作单位及职务", "联系电话"]
        },
        "才艺特长（请在sheet名称为“才艺特长”中填写）": {
            "sub_sheet": "才艺特长",
            "fields": ["才艺特长", "掌握程度"]
        }
    }

    # ---------------------- 2. 读取所有数据（自动适配引擎） ----------------------
    print("\n1. 读取数据（自动适配Excel引擎）...")
    def get_engine():
        pd_ver = pd.__version__.split('.')
        return 'openpyxl' if (int(pd_ver[0])>=1 and int(pd_ver[1])>=3) else 'xlrd'
    
    engine = get_engine()
    print(f"✅ 自动选择引擎：{engine}")

    # 读取主表（表单数据）
    try:
        form_data = pd.read_excel(
            excel_path, sheet_name="表单数据", 
            header=0, skiprows=[1], engine=engine
        )
        form_data.columns = [str(c).replace('\n', ' ').strip() for c in form_data.columns]
        form_data["学工号"] = form_data["学工号"].astype(str).str.strip()
        print(f"✅ 表单数据：{len(form_data)}条学生记录")
    except Exception as e:
        print(f"❌ 读取失败：{str(e)}")
        raise

    # 读取子表数据
    sub_data_dict = {}
    for main_col, cfg in sheet_mapping.items():
        try:
            sub_data = pd.read_excel(
                excel_path, sheet_name=cfg["sub_sheet"],
                header=0, skiprows=[1], engine=engine
            )
            sub_data.columns = [str(c).replace('\n', ' ').strip() for c in sub_data.columns]
            sub_data["学工号"] = sub_data["学工号"].astype(str).str.strip()
            sub_data = sub_data.dropna(subset=["学工号"])
            sub_data = sub_data[sub_data["学工号"] != "必填"]
            sub_data_dict[main_col] = sub_data
            print(f"✅ {cfg['sub_sheet']}：{len(sub_data)}条记录")
        except Exception as e:
            print(f"⚠️ {cfg['sub_sheet']}：无有效数据")
            sub_data_dict[main_col] = pd.DataFrame()

    # ---------------------- 3. 信息格式化函数 ----------------------
    def format_info(records, fields, info_type):
        if len(records) == 0:
            return f"无{info_type}记录"
        
        info_list = []
        for idx, (_, row) in enumerate(records.iterrows(), 1):
            if info_type == "受教育经历":
                info = f"[{idx}] 学校：{row.get(fields[0],'未知')}（{row.get(fields[1],'未知')}至{row.get(fields[2],'未知')}），专业：{row.get(fields[3],'无')}，学历：{row.get(fields[4],'无')}"
            elif info_type == "入学前主要获奖情况":
                info = f"[{idx}] 奖项：{row.get(fields[0],'未知')}（{row.get(fields[1],'未知级别')}），时间：{row.get(fields[2],'未知')}"
            elif info_type == "工作经历":
                info = f"[{idx}] 单位：{row.get(fields[0],'未知')}（{row.get(fields[1],'未知')}至{row.get(fields[2],'未知')}），职务：{row.get(fields[3],'未知')}，地点：{row.get(fields[4],'未知')}"
            elif info_type == "家庭成员信息":
                info = f"[{idx}] 姓名：{row.get(fields[0],'未知')}（{row.get(fields[1],'未知关系')}），工作：{row.get(fields[2],'无')}，电话：{row.get(fields[3],'无')}"
            elif info_type == "才艺特长":
                info = f"[{idx}] 特长：{row.get(fields[0],'未知')}，掌握程度：{row.get(fields[1],'一般')}"
            else:
                info = f"[{idx}] 未知信息"
            
            info_list.append(info)
        return "；".join(info_list)

    # ---------------------- 4. 全量补充信息 ----------------------
    print("\n2. 全量补充所有学生信息...")
    form_filled = form_data.copy()
    summary = []

    for main_col, cfg in sheet_mapping.items():
        sub_data = sub_data_dict[main_col]
        info_type = main_col.split("（")[0]
        
        if len(sub_data) == 0:
            form_filled[main_col] = f"无{info_type}记录"
            summary.append({"类型":info_type, "补充数":0, "状态":"无数据"})
            print(f"⚠️ {info_type}：无数据，统一填充默认文本")
            continue
        
        filled_cnt = 0
        for sid in set(form_filled["学工号"].dropna()):
            sid_records = sub_data[sub_data["学工号"] == sid]
            if len(sid_records) > 0:
                form_filled.loc[form_filled["学工号"]==sid, main_col] = format_info(sid_records, cfg["fields"], info_type)
                filled_cnt += 1
            else:
                form_filled.loc[form_filled["学工号"]==sid, main_col] = f"无{info_type}记录"
        
        summary.append({"类型":info_type, "补充数":filled_cnt, "状态":"完成"})
        print(f"✅ {info_type}：为{filled_cnt}名学生补充信息")

    # ---------------------- 5. 写入Excel + 统一格式（列宽+行高） ----------------------
    print(f"\n3. 写入Excel并统一格式（列宽={column_width}，行高={row_height}）...")
    try:
        # 加载工作簿
        wb = load_workbook(excel_path)
        ws = wb["表单数据"]
        
        # 1. 清空原有数据（保留前2行表头）
        if ws.max_row >=3:
            ws.delete_rows(3, ws.max_row-2)
        
        # 2. 获取Excel表头（第2行）
        excel_headers = []
        for col in ws.iter_cols(min_row=2, max_row=2, values_only=True):
            for val in col:
                excel_headers.append(str(val).replace('\n', ' ').strip() if val else "")
        
        # 3. 写入数据 + 设置单元格样式
        start_row = 3  # 数据从第3行开始
        for row_idx, (_, row_data) in enumerate(form_filled.iterrows(), start_row):
            for col_idx, (df_col, excel_col) in enumerate(zip(form_filled.columns, excel_headers)):
                # 写入单元格值
                cell_val = row_data[df_col] if not pd.isna(row_data[df_col]) else ""
                cell = ws.cell(row=row_idx, column=col_idx+1, value=cell_val)
                
                # 设置单元格样式（自动换行+垂直居中）
                cell.alignment = Alignment(
                    wrap_text=True,  # 自动换行（关键，避免内容溢出）
                    vertical='center',  # 垂直居中
                    horizontal='left'  # 水平居左（便于阅读）
                )
                cell.font = Font(name='微软雅黑', size=10)  # 统一字体
        
        # 4. 统一列宽（所有列）
        for col in range(1, ws.max_column+1):
            ws.column_dimensions[get_column_letter(col)].width = column_width
        
        # 5. 统一行高（表头行+数据行分开设置，表头行更高）
        # 第1-2行（表头/说明行）行高设为20
        ws.row_dimensions[1].height = 20
        ws.row_dimensions[2].height = 20
        # 第3行及以后（数据行）统一设为row_height
        for row in range(3, start_row + len(form_filled)):
            ws.row_dimensions[row].height = row_height
        
        # 保存文件
        wb.save(output_path)
        print(f"✅ 格式统一完成！文件保存至：{output_path}")

        # 生成报告
        report = f"""# 学生信息处理报告
## 处理结果
- 原始文件：{excel_path}
- 输出文件：{output_path}
- 处理时间：{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
- 学生总数：{len(form_filled)}人

## 格式规范
1. 列宽：所有列统一为 {column_width}
2. 行高：表头行（1-2行）20，数据行（3行起）统一为 {row_height}
3. 单元格样式：微软雅黑10号字，自动换行，垂直居中
4. 信息填充：所有学生5类信息全量补充，无记录显示“无XX信息记录”

## 补充详情
| 信息类型 | 补充学生数 | 状态 |
|----------|------------|------|"""
        for item in summary:
            report += f"\n| {item['类型']} | {item['补充数']} | {item['状态']} |"
        
        with open(output_path.replace(".xlsx", "_格式报告.md"), "w", encoding="utf-8") as f:
            f.write(report)
        print(f"✅ 报告生成完成！")

    except Exception as e:
        print(f"❌ 格式处理失败：{str(e)}")
        raise

    print("\n" + "="*70)
    print("🎉 所有操作完成！文件特性：")
    print(f"1. 信息：所有学生5类信息全量补充")
    print(f"2. 列宽：{column_width}（所有列完全一致）")
    print(f"3. 行高：数据行{row_height}（所有数据行完全一致）")
    print(f"4. 样式：自动换行+垂直居中，阅读体验优化")
    print("="*70)

    return form_filled

# ---------------------- 执行函数 ----------------------
# 配置参数（可根据需求调整）
INPUT_FILE = "D:/桌面/批量操作模板.xlsx"  # 原始文件路径
OUTPUT_FILE = "D:/桌面/批量操作模板_最终版.xlsx"  # 输出文件路径
UNIFORM_COL_WIDTH = 15  # 所有列统一宽度（建议12-20之间）
UNIFORM_ROW_HEIGHT = 30  # 所有数据行统一高度（建议25-40之间，根据内容调整）

# 运行（确保原始文件已关闭）
try:
    result_data = full_fill_format_all(
        excel_path=INPUT_FILE,
        output_path=OUTPUT_FILE,
        column_width=UNIFORM_COL_WIDTH,
        row_height=UNIFORM_ROW_HEIGHT
    )
    print(f"\n✅ 最终文件已生成：{OUTPUT_FILE}")
    print("👉 打开桌面即可看到，所有数据行高度完全一致！")
except Exception as e:
    print(f"\n❌ 执行失败：{str(e)}")
    print("💡 解决方案：1. 关闭Excel文件 2. 检查路径是否正确 3. 重新运行")

开始执行：信息全量补充 + 列宽统一(15) + 行高统一(30)

1. 读取数据（自动适配Excel引擎）...
✅ 自动选择引擎：xlrd
✅ 表单数据：363条学生记录
✅ 受教育经历（从高中开始）：451条记录
✅ 入学前主要获奖情况：97条记录
✅ 工作经历：27条记录
✅ 家庭成员信息：806条记录
✅ 才艺特长：46条记录

2. 全量补充所有学生信息...
✅ 受教育经历：为362名学生补充信息
✅ 入学前主要获奖情况：为57名学生补充信息
✅ 工作经历：为23名学生补充信息
✅ 家庭成员信息：为362名学生补充信息
✅ 才艺特长：为36名学生补充信息

3. 写入Excel并统一格式（列宽=15，行高=30）...
✅ 格式统一完成！文件保存至：D:/桌面/批量操作模板_最终版.xlsx
✅ 报告生成完成！

🎉 所有操作完成！文件特性：
1. 信息：所有学生5类信息全量补充
2. 列宽：15（所有列完全一致）
3. 行高：数据行30（所有数据行完全一致）
4. 样式：自动换行+垂直居中，阅读体验优化

✅ 最终文件已生成：D:/桌面/批量操作模板_最终版.xlsx
👉 打开桌面即可看到，所有数据行高度完全一致！


In [3]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

def custom_fill_format_excel(excel_path, output_path, data_row_height=30):
    """
    定制化功能：
    1. 保留第1行原始高度，第2行及以后统一高度（data_row_height）
    2. 受教育经历、家庭成员信息：合并后居中填写
    3. 信息精简：
       - 入学前获奖：只保留获奖名称
       - 工作经历：只保留任职单位
       - 才艺特长：只保留特长种类
    """
    print("="*70)
    print(f"开始定制化处理：保留表头高度+信息精简+居中对齐")
    print("="*70)

    # ---------------------- 1. 定制化数据表映射（按需求精简字段） ----------------------
    # key：表单数据列名；value：{子表名, 字段, 是否居中}
    sheet_mapping = {
        "受教育经历（从高中开始）（请在sheet名称为“受教育经历（从高中开始）”中填写）": {
            "sub_sheet": "受教育经历（从高中开始）",
            "fields": ["学校名称", "起始日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", 
                      "终止日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", "专业", "学历/学位"],
            "need_center": True,  # 需要居中填写
            "info_type": "受教育经历"
        },
        "入学前主要获奖情况（请在sheet名称为“入学前主要获奖情况”中填写）": {
            "sub_sheet": "入学前主要获奖情况",
            "fields": ["奖励名称"],  # 只保留获奖名称
            "need_center": False,
            "info_type": "入学前获奖"
        },
        "工作经历（请在sheet名称为“工作经历”中填写）": {
            "sub_sheet": "工作经历",
            "fields": ["任职单位"],  # 只保留任职单位
            "need_center": False,
            "info_type": "工作经历"
        },
        "家庭成员信息（请在sheet名称为“家庭成员信息”中填写）": {
            "sub_sheet": "家庭成员信息",
            "fields": ["姓名", "与本人关系", "工作单位及职务", "联系电话"],
            "need_center": True,  # 需要居中填写
            "info_type": "家庭成员信息"
        },
        "才艺特长（请在sheet名称为“才艺特长”中填写）": {
            "sub_sheet": "才艺特长",
            "fields": ["才艺特长"],  # 只保留特长种类
            "need_center": False,
            "info_type": "才艺特长"
        }
    }

    # ---------------------- 2. 读取数据（自动适配引擎） ----------------------
    print("\n1. 读取Excel数据...")
    def get_suitable_engine():
        """自动选择适配的Excel读取引擎"""
        pd_version = pd.__version__.split('.')
        return 'openpyxl' if (int(pd_version[0]) >= 1 and int(pd_version[1]) >= 3) else 'xlrd'
    
    engine = get_suitable_engine()
    print(f"✅ 自动适配引擎：{engine}")

    # 读取表单数据主表
    try:
        form_data = pd.read_excel(
            excel_path, sheet_name="表单数据",
            header=0, skiprows=[1], engine=engine
        )
        # 清理列名（去除换行符和多余空格）
        form_data.columns = [str(col).replace('\n', ' ').strip() for col in form_data.columns]
        # 确保学工号为字符串格式（关键匹配字段）
        form_data["学工号"] = form_data["学工号"].astype(str).str.strip()
        print(f"✅ 表单数据读取完成：共{len(form_data)}名学生")
    except Exception as e:
        print(f"❌ 表单数据读取失败：{str(e)}")
        raise

    # 读取各子表数据
    sub_sheet_data = {}
    for main_col, config in sheet_mapping.items():
        sub_sheet_name = config["sub_sheet"]
        try:
            sub_data = pd.read_excel(
                excel_path, sheet_name=sub_sheet_name,
                header=0, skiprows=[1], engine=engine
            )
            sub_data.columns = [str(col).replace('\n', ' ').strip() for col in sub_data.columns]
            sub_data["学工号"] = sub_data["学工号"].astype(str).str.strip()
            # 清理无效数据（去除学工号为空或“必填”的行）
            sub_data = sub_data.dropna(subset=["学工号"])
            sub_data = sub_data[sub_data["学工号"] != "必填"]
            sub_data = sub_data[sub_data["学工号"] != "nan"]
            
            sub_sheet_data[main_col] = sub_data
            print(f"✅ {sub_sheet_name}：{len(sub_data)}条有效记录")
        except Exception as e:
            print(f"⚠️ {sub_sheet_name}读取警告：{str(e)}（将填充默认文本）")
            sub_sheet_data[main_col] = pd.DataFrame()

    # ---------------------- 3. 定制化信息格式化函数（按需求精简） ----------------------
    def format_custom_info(records, fields, info_type):
        """
        按需求精简并格式化信息
        records: 某学生的对应信息记录
        fields: 需要保留的字段
        info_type: 信息类型名称
        """
        if len(records) == 0:
            return f"无{info_type}记录"
        
        info_list = []
        for idx, (_, row) in enumerate(records.iterrows(), 1):
            if info_type == "受教育经历":
                # 格式：[1] 学校：XX大学（2020-09至2024-06），专业：计算机，学历：本科
                school = str(row.get(fields[0], "未知学校")) if pd.notna(row.get(fields[0])) else "未知学校"
                start = str(row.get(fields[1], "未知")) if pd.notna(row.get(fields[1])) else "未知"
                end = str(row.get(fields[2], "未知")) if pd.notna(row.get(fields[2])) else "未知"
                major = str(row.get(fields[3], "无专业")) if pd.notna(row.get(fields[3])) else "无专业"
                degree = str(row.get(fields[4], "无学历")) if pd.notna(row.get(fields[4])) else "无学历"
                info_str = f"[{idx}] 学校：{school}（{start}至{end}），专业：{major}，学历：{degree}"
            
            elif info_type == "入学前获奖":
                # 格式：[1] 国家奖学金；[2] 省级三好学生（只保留获奖名称）
                award_name = str(row.get(fields[0], "未知奖项")) if pd.notna(row.get(fields[0])) else "未知奖项"
                info_str = f"[{idx}] {award_name}"
            
            elif info_type == "工作经历":
                # 格式：[1] XX科技公司；[2] YY企业（只保留任职单位）
                company = str(row.get(fields[0], "未知单位")) if pd.notna(row.get(fields[0])) else "未知单位"
                info_str = f"[{idx}] {company}"
            
            elif info_type == "家庭成员信息":
                # 格式：[1] 张三（父子）：XX工厂工人，电话13800138000
                name = str(row.get(fields[0], "未知姓名")) if pd.notna(row.get(fields[0])) else "未知姓名"
                relation = str(row.get(fields[1], "未知关系")) if pd.notna(row.get(fields[1])) else "未知关系"
                job = str(row.get(fields[2], "无工作信息")) if pd.notna(row.get(fields[2])) else "无工作信息"
                phone = str(row.get(fields[3], "无电话")) if pd.notna(row.get(fields[3])) else "无电话"
                info_str = f"[{idx}] {name}（{relation}）：{job}，电话{phone}"
            
            elif info_type == "才艺特长":
                # 格式：[1] 钢琴；[2] 篮球（只保留特长种类）
                talent = str(row.get(fields[0], "未知特长")) if pd.notna(row.get(fields[0])) else "未知特长"
                info_str = f"[{idx}] {talent}"
            
            else:
                info_str = f"[{idx}] 未知信息"
            
            info_list.append(info_str)
        
        return "；".join(info_list)

    # ---------------------- 4. 全量补充定制化信息 ----------------------
    print("\n2. 按需求补充定制化信息...")
    form_data_custom = form_data.copy()
    fill_summary = []

    for main_col, config in sheet_mapping.items():
        sub_data = sub_sheet_data[main_col]
        fields = config["fields"]
        info_type = config["info_type"]
        need_center = config["need_center"]
        
        # 统计补充情况
        filled_count = 0
        all_student_ids = set(form_data_custom["学工号"].dropna())
        
        for student_id in all_student_ids:
            # 筛选该学生的对应记录
            student_records = sub_data[sub_data["学工号"] == student_id] if len(sub_data) > 0 else pd.DataFrame()
            
            # 格式化信息
            if len(student_records) > 0:
                formatted_info = format_custom_info(student_records, fields, info_type)
                filled_count += 1
            else:
                formatted_info = f"无{info_type}记录"
            
            # 补充到表单数据
            form_data_custom.loc[form_data_custom["学工号"] == student_id, main_col] = formatted_info
        
        # 记录补充汇总
        fill_summary.append({
            "信息类型": info_type,
            "补充学生数": filled_count,
            "总原始记录数": len(sub_data),
            "是否居中": "是" if need_center else "否"
        })
        print(f"✅ {info_type}：补充{filled_count}名学生（居中：{'是' if need_center else '否'}）")

    # 显示补充汇总
    print("\n3. 信息补充汇总：")
    summary_df = pd.DataFrame(fill_summary)
    print(summary_df.to_string(index=False))

    # ---------------------- 5. 写入Excel并应用定制化格式 ----------------------
    print(f"\n4. 写入Excel并应用格式（第2行后统一行高：{data_row_height}）...")
    try:
        # 加载原始工作簿（保留第1行原始高度）
        wb = load_workbook(excel_path)
        ws = wb["表单数据"]
        
        # 记录第1行原始高度（关键：保留原始高度）
        original_row1_height = ws.row_dimensions[1].height
        print(f"✅ 保留第1行原始高度：{original_row1_height if original_row1_height else '默认'}")

        # 步骤1：清空原有数据（保留前2行表头/说明行）
        if ws.max_row >= 3:
            ws.delete_rows(3, ws.max_row - 2)
        print(f"✅ 清空原有数据行（保留前2行）")

        # 步骤2：获取Excel表头（第2行，用于匹配列顺序）
        excel_headers = []
        for col in ws.iter_cols(min_row=2, max_row=2, values_only=True):
            for cell_value in col:
                if cell_value:
                    excel_headers.append(str(cell_value).replace('\n', ' ').strip())
                else:
                    excel_headers.append("")
        
        # 步骤3：写入定制化补充后的数据 + 设置单元格样式
        data_start_row = 3  # 数据从第3行开始写入
        for row_idx, (_, row_data) in enumerate(form_data_custom.iterrows(), data_start_row):
            for col_idx, (df_col, excel_col) in enumerate(zip(form_data_custom.columns, excel_headers)):
                if df_col not in form_data_custom.columns:
                    continue
                
                # 获取单元格值（空值处理）
                cell_value = row_data[df_col]
                if pd.isna(cell_value):
                    cell_value = ""
                
                # 写入单元格
                cell = ws.cell(row=row_idx, column=col_idx + 1, value=cell_value)
                
                # 设置字体（统一为微软雅黑10号字）
                cell.font = Font(name='微软雅黑', size=10)
                
                # 判断是否需要居中（受教育经历、家庭成员信息居中）
                need_center = False
                for main_col, config in sheet_mapping.items():
                    if df_col == main_col and config["need_center"]:
                        need_center = True
                        break
                
                # 设置对齐方式
                if need_center:
                    # 需要居中的列：水平居中 + 垂直居中 + 自动换行
                    cell.alignment = Alignment(
                        horizontal='center',
                        vertical='center',
                        wrap_text=True
                    )
                else:
                    # 其他列：水平居左 + 垂直居中 + 自动换行
                    cell.alignment = Alignment(
                        horizontal='left',
                        vertical='center',
                        wrap_text=True
                    )

        # 步骤4：设置行高（核心需求）
        # 第1行：保留原始高度（不修改）
        # 第2行：统一设为20（作为表头补充行，与第1行区分）
        ws.row_dimensions[2].height = 20
        # 第3行及以后（所有数据行）：统一设为指定高度
        for row in range(3, data_start_row + len(form_data_custom)):
            ws.row_dimensions[row].height = data_row_height
        print(f"✅ 行高设置完成：第1行原始高度，第2行20，第3行后{data_row_height}")

        # 步骤5：统一列宽（保持所有列宽度一致，默认15，可调整）
        uniform_col_width = 15
        for col in range(1, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(col)].width = uniform_col_width
        print(f"✅ 列宽统一设置为：{uniform_col_width}")

        # 保存文件
        wb.save(output_path)
        print(f"✅ 定制化文件保存完成：{output_path}")

        # 生成定制化报告
        report_content = f"""# Excel定制化处理报告
## 处理配置
- 原始文件路径：{excel_path}
- 输出文件路径：{output_path}
- 处理时间：{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
- 学生总数：{len(form_data_custom)}名

## 核心格式（按需求定制）
1. 行高设置：
   - 第1行：保留原始高度（{original_row1_height if original_row1_height else '默认'}）
   - 第2行：统一20
   - 第3行及以后：统一{data_row_height}（所有数据行高度一致）
2. 列宽设置：所有列统一{uniform_col_width}
3. 对齐方式：
   - 受教育经历、家庭成员信息：水平居中+垂直居中
   - 其他列：水平居左+垂直居中
4. 字体：微软雅黑10号字（所有单元格统一）

## 信息精简规则
| 信息类型 | 保留字段 | 格式示例 |
|----------|----------|----------|
| 受教育经历 | 学校、时间、专业、学历 | [1] 学校：XX大学（2020-09至2024-06），专业：计算机，学历：本科 |
| 入学前获奖 | 仅获奖名称 | [1] 国家奖学金；[2] 省级三好学生 |
| 工作经历 | 仅任职单位 | [1] XX科技公司；[2] YY企业 |
| 家庭成员信息 | 姓名、关系、工作、电话 | [1] 张三（父子）：XX工厂工人，电话13800138000 |
| 才艺特长 | 仅特长种类 | [1] 钢琴；[2] 篮球 |

## 补充结果
| 信息类型 | 补充学生数 | 总原始记录数 | 是否居中 |
|----------|------------|--------------|----------|"""
        for item in fill_summary:
            report_content += f"\n| {item['信息类型']} | {item['补充学生数']} | {item['总原始记录数']} | {item['是否居中']} |"
        
        # 保存报告
        report_path = output_path.replace(".xlsx", "_定制化报告.md")
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"✅ 定制化报告生成完成：{report_path}")

    except Exception as e:
        print(f"❌ Excel格式处理失败：{str(e)}")
        raise

    # ---------------------- 6. 最终结果确认 ----------------------
    print("\n" + "="*70)
    print("🎉 定制化处理全部完成！最终效果：")
    print("1. 行高：第1行原始高度，第2行20，第3行后统一" + str(data_row_height))
    print("2. 信息：按需求精简（获奖只留名称、工作只留单位等）")
    print("3. 对齐：受教育经历、家庭成员信息居中填写")
    print("4. 格式：所有列宽15，微软雅黑10号字，自动换行")
    print("="*70)

    return form_data_custom

# ---------------------- 执行定制化处理 ----------------------
# 配置文件路径（请确认你的文件在D盘桌面）
INPUT_EXCEL = "D:/桌面/批量操作模板.xlsx"  # 原始文件路径
OUTPUT_EXCEL = "D:/桌面/批量操作模板_定制化最终版.xlsx"  # 输出文件路径
DATA_ROW_HEIGHT = 30  # 第2行后统一行高（可根据需求调整，建议25-40）

# 执行（确保原始Excel文件已关闭）
try:
    result_data = custom_fill_format_excel(
        excel_path=INPUT_EXCEL,
        output_path=OUTPUT_EXCEL,
        data_row_height=DATA_ROW_HEIGHT
    )
    print(f"\n✅ 最终文件已生成：{OUTPUT_EXCEL}")
    print("👉 打开D盘桌面即可查看，完全符合你的定制需求！")
except Exception as e:
    print(f"\n❌ 执行失败：{str(e)}")
    print("💡 解决方案：1. 关闭所有Excel文件 2. 检查文件路径是否正确 3. 重新运行")

开始定制化处理：保留表头高度+信息精简+居中对齐

1. 读取Excel数据...
✅ 自动适配引擎：xlrd
✅ 表单数据读取完成：共363名学生
✅ 受教育经历（从高中开始）：451条有效记录
✅ 入学前主要获奖情况：97条有效记录
✅ 工作经历：27条有效记录
✅ 家庭成员信息：806条有效记录
✅ 才艺特长：46条有效记录

2. 按需求补充定制化信息...
✅ 受教育经历：补充362名学生（居中：是）
✅ 入学前获奖：补充57名学生（居中：否）
✅ 工作经历：补充23名学生（居中：否）
✅ 家庭成员信息：补充362名学生（居中：是）
✅ 才艺特长：补充36名学生（居中：否）

3. 信息补充汇总：
   信息类型  总原始记录数 是否居中  补充学生数
  受教育经历     451    是    362
  入学前获奖      97    否     57
   工作经历      27    否     23
 家庭成员信息     806    是    362
   才艺特长      46    否     36

4. 写入Excel并应用格式（第2行后统一行高：30）...
✅ 保留第1行原始高度：默认
✅ 清空原有数据行（保留前2行）
✅ 行高设置完成：第1行原始高度，第2行20，第3行后30
✅ 列宽统一设置为：15
✅ 定制化文件保存完成：D:/桌面/批量操作模板_定制化最终版.xlsx
✅ 定制化报告生成完成：D:/桌面/批量操作模板_定制化最终版_定制化报告.md

🎉 定制化处理全部完成！最终效果：
1. 行高：第1行原始高度，第2行20，第3行后统一30
2. 信息：按需求精简（获奖只留名称、工作只留单位等）
3. 对齐：受教育经历、家庭成员信息居中填写
4. 格式：所有列宽15，微软雅黑10号字，自动换行

✅ 最终文件已生成：D:/桌面/批量操作模板_定制化最终版.xlsx
👉 打开D盘桌面即可查看，完全符合你的定制需求！


In [4]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

def ultra_simple_format_excel(excel_path, output_path, data_row_height=30):
    """
    极致精简格式：
    1. 完全去除[1]序号标识，仅保留纯内容
    2. 受教育经历：直接显示“学校（时间）专业/学历”，无“学校：”前缀
    3. 入学前获奖：仅显示获奖名称（纯文本，无任何前缀）
    4. 工作经历：仅显示任职单位（纯文本）
    5. 家庭成员：仅保留“称谓、姓名、工作单位、职务、政治面貌、电话”
    6. 保留行高规则：第1行原始高度，第2行后统一高度
    """
    print("="*70)
    print("开始执行：极致精简格式优化（去除序号+保留核心字段）")
    print("="*70)

    # ---------------------- 1. 核心字段映射（按需求精准筛选） ----------------------
    sheet_mapping = {
        "受教育经历（从高中开始）（请在sheet名称为“受教育经历（从高中开始）”中填写）": {
            "sub_sheet": "受教育经历（从高中开始）",
            "fields": ["学校名称", "起始日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", 
                      "终止日期 支持日期格式：yyyy-MM-dd、yyyy/MM/dd 或 m/d/yyyy", "专业", "学历/学位"],
            "need_center": True,
            "info_type": "受教育经历"
        },
        "入学前主要获奖情况（请在sheet名称为“入学前主要获奖情况”中填写）": {
            "sub_sheet": "入学前主要获奖情况",
            "fields": ["奖励名称"],  # 仅保留获奖名称
            "need_center": False,
            "info_type": "入学前获奖"
        },
        "工作经历（请在sheet名称为“工作经历”中填写）": {
            "sub_sheet": "工作经历",
            "fields": ["任职单位"],  # 仅保留任职单位
            "need_center": False,
            "info_type": "工作经历"
        },
        "家庭成员信息（请在sheet名称为“家庭成员信息”中填写）": {
            "sub_sheet": "家庭成员信息",
            "fields": ["与本人关系", "姓名", "工作单位及职务", "职务", "政治面貌", "联系电话"],  # 核心6字段
            "need_center": True,
            "info_type": "家庭成员信息"
        },
        "才艺特长（请在sheet名称为“才艺特长”中填写）": {
            "sub_sheet": "才艺特长",
            "fields": ["才艺特长"],  # 仅保留特长种类
            "need_center": False,
            "info_type": "才艺特长"
        }
    }

    # ---------------------- 2. 读取数据（自动适配引擎） ----------------------
    print("\n1. 读取Excel数据...")
    def get_engine():
        pd_ver = pd.__version__.split('.')
        return 'openpyxl' if (int(pd_ver[0])>=1 and int(pd_ver[1])>=3) else 'xlrd'
    
    engine = get_engine()
    print(f"✅ 自动适配引擎：{engine}")

    # 读取表单数据主表
    try:
        form_data = pd.read_excel(
            excel_path, sheet_name="表单数据",
            header=0, skiprows=[1], engine=engine
        )
        form_data.columns = [str(col).replace('\n', ' ').strip() for col in form_data.columns]
        form_data["学工号"] = form_data["学工号"].astype(str).str.strip()
        print(f"✅ 表单数据：{len(form_data)}名学生")
    except Exception as e:
        print(f"❌ 读取失败：{str(e)}")
        raise

    # 读取子表数据
    sub_data_dict = {}
    for main_col, cfg in sheet_mapping.items():
        try:
            sub_data = pd.read_excel(
                excel_path, sheet_name=cfg["sub_sheet"],
                header=0, skiprows=[1], engine=engine
            )
            sub_data.columns = [str(col).replace('\n', ' ').strip() for col in sub_data.columns]
            sub_data["学工号"] = sub_data["学工号"].astype(str).str.strip()
            # 清理无效数据
            sub_data = sub_data.dropna(subset=["学工号"])
            sub_data = sub_data[sub_data["学工号"] != "必填"]
            sub_data_dict[main_col] = sub_data
            print(f"✅ {cfg['sub_sheet']}：{len(sub_data)}条记录")
        except Exception as e:
            print(f"⚠️ {cfg['sub_sheet']}：无有效数据（{str(e)}）")
            sub_data_dict[main_col] = pd.DataFrame()

    # ---------------------- 3. 极致精简格式化函数（完全去除序号） ----------------------
    def format_ultra_simple(records, fields, info_type):
        """完全去除序号，仅保留核心内容"""
        if len(records) == 0:
            return f"无{info_type}记录"
        
        content_list = []
        for _, row in records.iterrows():  # 不生成序号，直接循环
            if info_type == "受教育经历":
                # 格式：XX学校（2020-09至2024-06）计算机专业/本科学历
                school = str(row.get(fields[0], "未知学校")) if pd.notna(row.get(fields[0])) else "未知学校"
                start = str(row.get(fields[1], "")) if pd.notna(row.get(fields[1])) else ""
                end = str(row.get(fields[2], "")) if pd.notna(row.get(fields[2])) else ""
                time_str = f"（{start}-{end}）" if start or end else ""  # 无时间则不显示括号
                major = str(row.get(fields[3], "")) if pd.notna(row.get(fields[3])) else ""
                degree = str(row.get(fields[4], "")) if pd.notna(row.get(fields[4])) else ""
                major_degree = f"{major}专业/{degree}学历" if major or degree else ""
                content = f"{school}{time_str} {major_degree}".strip()
                content_list.append(content)
            
            elif info_type == "入学前获奖":
                # 格式：国家奖学金；省级三好学生（纯名称，无任何前缀）
                award = str(row.get(fields[0], "未知奖项")) if pd.notna(row.get(fields[0])) else "未知奖项"
                content_list.append(award)
            
            elif info_type == "工作经历":
                # 格式：世纪恒通科技股份有限公司；贵州黔聚标注科技有限公司（纯单位名）
                company = str(row.get(fields[0], "未知单位")) if pd.notna(row.get(fields[0])) else "未知单位"
                content_list.append(company)
            
            elif info_type == "家庭成员信息":
                # 格式：父子-张三-XX工厂-工人-群众-13800138000（核心6字段，用“-”分隔）
                relation = str(row.get(fields[0], "未知")) if pd.notna(row.get(fields[0])) else "未知"  # 称谓
                name = str(row.get(fields[1], "未知")) if pd.notna(row.get(fields[1])) else "未知"        # 姓名
                work_unit = str(row.get(fields[2], "无")) if pd.notna(row.get(fields[2])) else "无"      # 工作单位
                duty = str(row.get(fields[3], "无")) if pd.notna(row.get(fields[3])) else "无"          # 职务
                political = str(row.get(fields[4], "无")) if pd.notna(row.get(fields[4])) else "无"      # 政治面貌
                phone = str(row.get(fields[5], "无")) if pd.notna(row.get(fields[5])) else "无"          # 电话
                # 拼接为“称谓-姓名-工作单位-职务-政治面貌-电话”
                content = f"{relation}-{name}-{work_unit}-{duty}-{political}-{phone}"
                content_list.append(content)
            
            elif info_type == "才艺特长":
                # 格式：钢琴；篮球；编程（纯特长名）
                talent = str(row.get(fields[0], "未知特长")) if pd.notna(row.get(fields[0])) else "未知特长"
                content_list.append(talent)
        
        # 用“；”分隔多条记录，无序号
        return "；".join(content_list)

    # ---------------------- 4. 全量补充精简信息 ----------------------
    print("\n2. 补充极致精简格式的信息...")
    form_simple = form_data.copy()
    summary = []

    for main_col, cfg in sheet_mapping.items():
        sub_data = sub_data_dict[main_col]
        info_type = cfg["info_type"]
        fields = cfg["fields"]
        
        filled_cnt = 0
        all_sids = set(form_simple["学工号"].dropna())
        
        for sid in all_sids:
            sid_records = sub_data[sub_data["学工号"] == sid] if len(sub_data) > 0 else pd.DataFrame()
            
            if len(sid_records) > 0:
                simple_content = format_ultra_simple(sid_records, fields, info_type)
                filled_cnt += 1
            else:
                simple_content = f"无{info_type}记录"
            
            form_simple.loc[form_simple["学工号"] == sid, main_col] = simple_content
        
        summary.append({
            "信息类型": info_type,
            "补充学生数": filled_cnt,
            "总记录数": len(sub_data),
            "状态": "完成" if len(sub_data) > 0 else "无数据"
        })
        print(f"✅ {info_type}：补充{filled_cnt}名学生（无序号格式）")

    # 显示汇总
    print("\n3. 补充汇总：")
    print(pd.DataFrame(summary).to_string(index=False))

    # ---------------------- 5. 写入Excel并保持格式规则 ----------------------
    print(f"\n4. 写入Excel（第2行后统一行高：{data_row_height}）...")
    try:
        # 加载工作簿（保留第1行原始高度）
        wb = load_workbook(excel_path)
        ws = wb["表单数据"]
        original_row1_h = ws.row_dimensions[1].height
        print(f"✅ 保留第1行原始高度：{original_row1_h if original_row1_h else '默认'}")

        # 清空原有数据（保留前2行）
        if ws.max_row >=3:
            ws.delete_rows(3, ws.max_row-2)

        # 获取Excel表头（第2行）
        excel_headers = []
        for col in ws.iter_cols(min_row=2, max_row=2, values_only=True):
            for val in col:
                excel_headers.append(str(val).replace('\n', ' ').strip() if val else "")

        # 写入数据 + 设置样式
        start_row = 3
        for row_idx, (_, row_data) in enumerate(form_simple.iterrows(), start_row):
            for col_idx, (df_col, excel_col) in enumerate(zip(form_simple.columns, excel_headers)):
                if df_col not in form_simple.columns:
                    continue
                
                # 处理单元格值
                cell_val = row_data[df_col] if not pd.isna(row_data[df_col]) else ""
                cell = ws.cell(row=row_idx, column=col_idx+1, value=cell_val)
                
                # 统一字体
                cell.font = Font(name='微软雅黑', size=10)
                
                # 判断是否需要居中（受教育经历、家庭成员）
                need_center = any([df_col == mc and sheet_mapping[mc]["need_center"] for mc in sheet_mapping.keys()])
                cell.alignment = Alignment(
                    horizontal='center' if need_center else 'left',
                    vertical='center',
                    wrap_text=True
                )

        # 设置行高
        ws.row_dimensions[2].height = 20  # 第2行统一20
        for row in range(3, start_row + len(form_simple)):
            ws.row_dimensions[row].height = data_row_height  # 数据行统一高度

        # 统一列宽（15）
        col_width = 15
        for col in range(1, ws.max_column+1):
            ws.column_dimensions[get_column_letter(col)].width = col_width

        # 保存文件
        wb.save(output_path)
        print(f"✅ 极致精简格式文件保存：{output_path}")

        # 生成报告
        report = f"""# 极致精简格式处理报告
## 格式特点
1. 完全去除[1]序号标识，仅保留纯内容
2. 受教育经历：XX学校（时间）专业/学历（无“学校：”前缀）
3. 入学前获奖：仅显示获奖名称（如“国家奖学金；省级三好学生”）
4. 工作经历：仅显示任职单位（如“世纪恒通科技股份有限公司”）
5. 家庭成员：称谓-姓名-工作单位-职务-政治面貌-电话（用“-”分隔）
6. 行高：第1行原始高度，第2行20，第3行后{data_row_height}
7. 列宽：所有列统一{col_width}

## 处理结果
- 原始文件：{excel_path}
- 输出文件：{output_path}
- 学生总数：{len(form_simple)}人
- 处理时间：{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## 补充详情
| 信息类型 | 补充学生数 | 总记录数 | 状态 |
|----------|------------|----------|------|"""
        for item in summary:
            report += f"\n| {item['信息类型']} | {item['补充学生数']} | {item['总记录数']} | {item['状态']} |"
        
        with open(output_path.replace(".xlsx", "_精简报告.md"), "w", encoding="utf-8") as f:
            f.write(report)
        print(f"✅ 精简报告生成完成！")

    except Exception as e:
        print(f"❌ 写入失败：{str(e)}")
        raise

    print("\n" + "="*70)
    print("🎉 极致精简格式处理完成！")
    print("核心效果：无序号、无多余前缀、仅保留核心内容")
    print("="*70)

    return form_simple

# ---------------------- 执行函数 ----------------------
# 文件路径配置
INPUT_FILE = "D:/桌面/批量操作模板.xlsx"  # 原始文件
OUTPUT_FILE = "D:/桌面/批量操作模板_极致精简版.xlsx"  # 输出文件
DATA_ROW_HEIGHT = 30  # 数据行统一高度（可调整）

# 运行（确保原始文件已关闭）
try:
    result = ultra_simple_format_excel(
        excel_path=INPUT_FILE,
        output_path=OUTPUT_FILE,
        data_row_height=DATA_ROW_HEIGHT
    )
    print(f"\n✅ 最终文件：{OUTPUT_FILE}")
    print("👉 打开D盘桌面查看，格式完全符合你的要求！")
except Exception as e:
    print(f"\n❌ 执行失败：{str(e)}")
    print("💡 解决方案：1. 关闭Excel文件 2. 检查路径 3. 重新运行")

开始执行：极致精简格式优化（去除序号+保留核心字段）

1. 读取Excel数据...
✅ 自动适配引擎：xlrd
✅ 表单数据：363名学生
✅ 受教育经历（从高中开始）：451条记录
✅ 入学前主要获奖情况：97条记录
✅ 工作经历：27条记录
✅ 家庭成员信息：806条记录
✅ 才艺特长：46条记录

2. 补充极致精简格式的信息...
✅ 受教育经历：补充362名学生（无序号格式）
✅ 入学前获奖：补充57名学生（无序号格式）
✅ 工作经历：补充23名学生（无序号格式）
✅ 家庭成员信息：补充362名学生（无序号格式）
✅ 才艺特长：补充36名学生（无序号格式）

3. 补充汇总：
   信息类型  总记录数  状态  补充学生数
  受教育经历   451  完成    362
  入学前获奖    97  完成     57
   工作经历    27  完成     23
 家庭成员信息   806  完成    362
   才艺特长    46  完成     36

4. 写入Excel（第2行后统一行高：30）...
✅ 保留第1行原始高度：默认
✅ 极致精简格式文件保存：D:/桌面/批量操作模板_极致精简版.xlsx
✅ 精简报告生成完成！

🎉 极致精简格式处理完成！
核心效果：无序号、无多余前缀、仅保留核心内容

✅ 最终文件：D:/桌面/批量操作模板_极致精简版.xlsx
👉 打开D盘桌面查看，格式完全符合你的要求！
